# 01. Correlational analysis: what patterns do we observe?

Notebook 00 built the dataset. The causal question has not changed.

> **To what extent does applying the backdoor defense, instead of baseline random filtering, change the probability of successful backdoor detection?**

This notebook does not answer it. It measures association, meaning which variables move together, so that you can ask sharper questions when you build the causal graph in notebook 02.

> **Association**, also called correlation, means two variables tend to move together, so knowing one tells you something about the other.
>
> **Causation** means changing one makes the other change.

You can compute association from data. You cannot compute causation from data alone, because it depends on assumptions about how the data came about. Every number in this notebook is an association.

**Tutorial path:** 00 Data preparation, then **01 Correlational analysis**, then 02 Causal inference


## 1. Configure the exploration

These settings choose the correlation measure, the plot colors, and the sample size for the large pairplot.

We use Spearman correlation, which measures whether two variables move in the same direction without assuming the relationship follows a straight line. It runs from -1, where one variable rises as the other falls, through 0, where they share no consistent direction, up to +1, where they rise together.

Two settings matter more than the rest: `pre_treatment_variables` and `post_treatment_variables`. They record when each variable came into existence. You cannot learn that from the data. It comes from knowing how the pipeline works.


In [ ]:
def default_params():
    return {
        "causal_dataset": "data/causal_data.csv",
        "dag_worksheet_output": "data/dag_worksheet.csv",
        "treatment_column": "treatment",
        "outcome_column": "outcome",
        "covariate_columns": ['code_number_tokens', 'code_complexity', 'code_num_identifiers', 'code_num_strings', 'reviewer_experience', 'rollout_eligibility', 'noise_feature', 'inspection_intensity', 'manual_review_flag'],
        "pre_treatment_variables": ['code_number_tokens', 'code_complexity', 'code_num_identifiers', 'code_num_strings', 'reviewer_experience', 'rollout_eligibility', 'noise_feature'],
        "post_treatment_variables": ['inspection_intensity', 'manual_review_flag'],
        "correlation_method": "spearman",
        "plot_palette": "mako",
        "heatmap_palette": "vlag",
        "top_variables_to_plot": 7,
        "pairplot_max_rows": 3000,
        "random_seed": 42,
    }

params = default_params()
params


## 2. Load the dataset from notebook 00

Load the observed table: treatment, outcome, and covariates, one row per code example.


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from src.correlational_analysis_utils import (
    build_dag_worksheet,
    correlation_matrix,
    covariate_quality_table,
    load_causal_dataset,
    outcome_by_treatment_table,
    treatment_balance_table,
    variable_association_table,
)

sns.set_theme(style="whitegrid")

analysis_df, columns = load_causal_dataset(params)

treatment = columns.treatment
outcome = columns.outcome
covariates = columns.covariates

print(f"Rows: {len(analysis_df):,}")
print(f"Treatment column: {treatment}")
print(f"Outcome column: {outcome}")
print(f"Covariates: {len(covariates)}")

analysis_df.head()


## 3. Recall what treatment and outcome mean

Once every variable is a 0 or a 1, the domain meaning slips away easily.

| Value | Meaning |
|---|---|
| `treatment = 0` | random filtering baseline |
| `treatment = 1` | backdoor defense applied |
| `outcome = 0` | detection failed |
| `outcome = 1` | detection succeeded |

The mean of `outcome` within a group gives that group's detection success rate.


## 4. Check that each covariate varies

A variable that holds the same value for every unit is constant. A constant cannot correlate with anything, and it cannot distinguish treated units from control units.

Flag those variables now, so that the empty (`NaN`) correlations they produce later do not look like real findings.


In [ ]:
quality_table = covariate_quality_table(
    analysis_df,
    covariates,
)

informative_covariates = (
    quality_table
    .loc[quality_table["has_variation"], "variable"]
    .tolist()
)

constant_covariates = (
    quality_table
    .loc[~quality_table["has_variation"], "variable"]
    .tolist()
)

print(f"Varying covariates: {len(informative_covariates)}")
print(f"Constant covariates: {constant_covariates or 'none'}")

quality_table


## 5. Compare detection success across the two groups

Start with the simplest comparison: detection success among treated units against detection success among control units.

This gives you a descriptive difference, not the causal effect. The two groups did not form at random, so units that received the defense may differ from units that did not, in ways that also affect detection.


In [ ]:
outcome_summary = outcome_by_treatment_table(
    analysis_df,
    treatment=treatment,
    outcome=outcome,
)

outcome_summary["condition"] = outcome_summary[treatment].map(
    {
        0: "Random filtering",
        1: "Backdoor defense",
    }
)

control_dsr = float(
    outcome_summary.loc[outcome_summary[treatment] == 0, "outcome_mean"].iloc[0]
)
defense_dsr = float(
    outcome_summary.loc[outcome_summary[treatment] == 1, "outcome_mean"].iloc[0]
)

print(f"Random-filtering DSR: {control_dsr:.3f}")
print(f"Backdoor-defense DSR: {defense_dsr:.3f}")
print(f"Raw DSR difference:   {defense_dsr - control_dsr:.3f}")

outcome_summary


In [ ]:
plot_df = analysis_df.assign(
    treatment_condition=analysis_df[treatment].map(
        {
            0: "Random filtering",
            1: "Backdoor defense",
        }
    )
)

plt.figure(figsize=(7, 4))

sns.barplot(
    data=plot_df,
    x="treatment_condition",
    y=outcome,
    hue="treatment_condition",
    palette=params["plot_palette"],
    errorbar=("ci", 95),
    legend=False,
)

plt.title("Observed detection success by treatment condition")
plt.xlabel("")
plt.ylabel("Detection success rate")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()


### Why the raw difference is not enough

Suppose more complex code is both:

1. more likely to receive the backdoor defense, and
2. harder to detect successfully.

Complexity then pushes the two groups apart on its own. The treated group looks worse partly because it holds the harder cases, not only because of the defense. The gap you just measured mixes the two explanations together, and that single number cannot separate them.

> A **confounder** is a variable that causes both the treatment and the outcome. It produces an association between them that is not a causal effect.

Complexity in this example is a confounder. It affects which examples receive the defense, and it affects whether detection succeeds.

Confounding is the main reason a raw group difference misleads you. Handling it properly is what notebook 02 is about.


## 6. Look at pairwise correlations

The heatmap shows the association between every pair of variables. Use it to spot:

- variables related to the treatment;
- variables related to the outcome;
- clusters of variables that measure nearly the same thing;
- possible proxies, meaning variables that carry no causal force of their own but stand in for something that does.

Do not read arrow directions off this heatmap. Correlation is symmetric, so the correlation of A with B equals the correlation of B with A. Causation is not symmetric. The heatmap cannot tell you which way an arrow points, or whether an arrow belongs there at all.


In [ ]:
analysis_columns = [
    treatment,
    outcome,
    *informative_covariates,
]

corr = correlation_matrix(
    analysis_df,
    columns=analysis_columns,
    method=params["correlation_method"],
)

corr.round(2)


In [ ]:
mask = np.triu(
    np.ones_like(corr, dtype=bool),
    k=1,
)

plt.figure(figsize=(12, 9))

sns.heatmap(
    corr,
    mask=mask,
    cmap=params["heatmap_palette"],
    center=0,
    vmin=-1,
    vmax=1,
    annot=True,
    fmt=".2f",
    square=True,
    linewidths=0.5,
    cbar_kws={
        "label": f"{params['correlation_method'].title()} correlation"
    },
)

plt.title("Pairwise associations among observed variables")
plt.tight_layout()
plt.show()


## 7. Which variables differ between the treatment groups?

The standardized mean difference, or SMD, compares a variable's average in the treated group against its average in the control group. It reports the gap in standard deviations, so variables measured on different scales can sit in the same plot.

- A value near 0 means the two groups look alike on that variable.
- A value far from 0 means the groups are imbalanced on it.

Imbalance gives you a clue about how the pipeline assigned treatment. It does not prove confounding. A variable that the pipeline measured after treatment can be just as imbalanced, precisely because the treatment changed it. The size of an imbalance tells you nothing about what came first.


In [ ]:
balance_table = treatment_balance_table(
    analysis_df,
    treatment=treatment,
    covariates=covariates,
)

balance_table.round(3)


In [ ]:
balance_plot = (
    balance_table
    .loc[balance_table["has_variation"]]
    .sort_values("standardized_mean_difference")
)

plt.figure(figsize=(9, 6))

sns.barplot(
    data=balance_plot,
    x="standardized_mean_difference",
    y="variable",
    hue="variable",
    palette=params["plot_palette"],
    legend=False,
)

plt.axvline(0, color="black", linewidth=1)
plt.axvline(-0.10, color="gray", linestyle="--", linewidth=1)
plt.axvline(0.10, color="gray", linestyle="--", linewidth=1)

plt.title("Covariate imbalance: backdoor defense vs random filtering")
plt.xlabel("Standardized mean difference")
plt.ylabel("")
plt.tight_layout()
plt.show()


## 8. Plot association with treatment against association with outcome

This helps you decide which variables deserve discussion.

A variable associated with both the treatment and the outcome deserves attention, but that one pattern can arise from very different structures.

| Structure | What it means | Shape |
|---|---|---|
| **Confounder** | causes both | treatment ← X → outcome |
| **Mediator** | the treatment causes it, and it causes the outcome, so it carries part of the effect | treatment → X → outcome |
| **Collider** | both the treatment and the outcome cause it | treatment → X ← outcome |
| **Proxy** | stands in for another variable without acting on its own | |

All four can land a variable in the same position on this plot. Telling them apart takes knowledge of mechanism and timing, not a larger correlation.

The distinction matters in practice, because each one calls for different handling. Adjusting for a confounder removes bias. Adjusting for a mediator removes part of the very effect you want to measure. Adjusting for a collider creates bias where none existed.


In [ ]:
association_table = variable_association_table(
    analysis_df,
    treatment=treatment,
    outcome=outcome,
    covariates=informative_covariates,
    method=params["correlation_method"],
)

association_table.round(3)


In [ ]:
plt.figure(figsize=(9, 7))

ax = sns.scatterplot(
    data=association_table,
    x="association_with_treatment",
    y="association_with_outcome",
    size="screening_score",
    hue="screening_score",
    palette=params["plot_palette"],
    sizes=(70, 320),
    legend=False,
)

plt.axvline(0, color="gray", linewidth=1)
plt.axhline(0, color="gray", linewidth=1)

for row in association_table.itertuples():
    ax.text(
        row.association_with_treatment + 0.008,
        row.association_with_outcome + 0.008,
        row.variable,
        fontsize=9,
    )

plt.title("Observed association with treatment and detection outcome")
plt.xlabel(f"Association with treatment ({params['correlation_method']})")
plt.ylabel(f"Association with outcome ({params['correlation_method']})")
plt.tight_layout()
plt.show()


## 9. Add timing before you assign causal roles

This is the key step of the notebook.

You compute correlation from the data. Timing comes from knowing how the data came about, and timing rules out entire categories of causal role.

Recall the pipeline each example passed through.

```text
  code features, reviewer_experience,       these exist before anything happens
  rollout_eligibility, noise_feature
              |
              v
       a method is chosen                   treatment
              |
              v
     the method does its work               inspection_intensity
              |
              v
       the verdict is recorded              outcome
              |
              v
       a human may review it                manual_review_flag
```

The code features, `reviewer_experience`, `rollout_eligibility` and `noise_feature` all hold their values before treatment. Any of them can be a confounder, because each one already existed when the pipeline chose a method, so each was in a position to influence that choice.

`inspection_intensity` and `manual_review_flag` come into existence after treatment. Neither can be a confounder, whatever the correlations say. A confounder has to cause the treatment, and nothing that appears after the pipeline chose a method can have caused that choice. They may be mediators or colliders instead.

Notice that this argument uses no statistics at all. It rests entirely on the order of the stages. That is the point. Knowing how the data came about settles some causal questions, and no amount of correlation substitutes for it.


In [ ]:
timing_map = {
    **{
        variable: "before treatment"
        for variable in params["pre_treatment_variables"]
    },
    **{
        variable: "after treatment"
        for variable in params["post_treatment_variables"]
    },
}

pd.DataFrame(
    {
        "variable": covariates,
        "measurement_timing": [timing_map[v] for v in covariates],
    }
)


## 10. Inspect a few strong relationships closely

A correlation coefficient compresses a whole relationship into one number, which can hide curvature, separate clusters, and how much the two treatment groups overlap.

For readability this cell plots only the top few variables, on a sample of at most `pairplot_max_rows` rows.


In [ ]:
top_n = min(
    int(params["top_variables_to_plot"]),
    len(association_table),
)

top_variables = association_table.head(top_n)["variable"].tolist()
pairplot_variables = top_variables[: min(4, len(top_variables))]
pairplot_n = min(int(params["pairplot_max_rows"]), len(analysis_df))

if pairplot_variables:
    pairplot_df = (
        analysis_df[[treatment, *pairplot_variables]]
        .sample(
            n=pairplot_n,
            random_state=params["random_seed"],
        )
    )

    sns.pairplot(
        pairplot_df,
        vars=pairplot_variables,
        hue=treatment,
        palette=sns.color_palette("colorblind", n_colors=2),
        corner=True,
        diag_kind="hist",
        plot_kws={
            "alpha": 0.50,
            "s": 22,
        },
    )
    plt.show()
else:
    print("No varying covariates available for the pairplot.")


## 11. Build the DAG worksheet

The worksheet is what you carry into notebook 02. It combines:

- association with the treatment;
- association with the outcome;
- imbalance between the treatment groups;
- when each variable came into existence;
- blank columns for your own causal reasoning.

> A **DAG**, short for directed acyclic graph, diagrams your causal assumptions. Each variable is a node, and each arrow `A → B` claims that A directly causes B. It is directed because arrows point one way, and acyclic because you can never follow arrows and arrive back where you started.

Fill the blank columns by reasoning about mechanism and timing, not by ranking correlations.


In [ ]:
dag_worksheet = build_dag_worksheet(
    association_table=association_table,
    balance_table=balance_table,
    quality_table=quality_table,
    treatment=treatment,
    outcome=outcome,
    timing_map=timing_map,
)

dag_worksheet


## 12. Answer these before notebook 02

For each variable, ask:

1. Did it hold its value before treatment, or did it appear afterwards?
2. Could it cause the pipeline to apply the defense?
3. Could it cause detection to succeed?
4. Could the treatment cause it?
5. Could the outcome cause it?
6. Does it mainly stand in for another variable?
7. Would adjusting for it block part of the treatment effect, making it a mediator?
8. Could adjusting for it open an unwanted path, making it a collider?
9. What mechanism justifies each arrow you propose?

No single correct answer exists for you to copy out. You want a set of assumptions you can state plainly and defend.


In [ ]:
from pathlib import Path

worksheet_path = Path(params["dag_worksheet_output"])
worksheet_path.parent.mkdir(parents=True, exist_ok=True)
dag_worksheet.to_csv(worksheet_path, index=False)

print(f"Saved DAG worksheet to: {worksheet_path}")


## Next: notebook 02

Notebook 01 answered one question: what patterns appear in the observed data?

Notebook 02 asks a harder one: what causal structure could have produced those patterns, and what does it imply about the effect of treatment on outcome?

Take the DAG worksheet with you.
